# Data Exploration

Explore the GSM8K mathematical word-problem dataset.

## Load the dataset

Read the downloaded dataset and inspect its splits and features.

In [0]:

from pathlib import Path
from collections import Counter
from itertools import combinations
from decimal import Decimal, InvalidOperation
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from IPython.display import display

STORAGE_DIR = Path("/Volumes/workspace/default/numreason")
DATA_DIR = STORAGE_DIR / "data_split"
MODEL_DIR = STORAGE_DIR / "models" / "Qwen3.5-0.8B"

if not (DATA_DIR / "dataset_dict.json").is_file():
    raise FileNotFoundError(f"Saved dataset not found at {DATA_DIR}. Run preprocess_data.py first.")

dataset = load_from_disk(str(DATA_DIR))
train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(dataset)
print("\nTraining features:")
print(train.features)

display(pd.DataFrame({
    "Split": list(dataset.keys()),
    "Examples": [len(dataset[name]) for name in dataset.keys()],
}))

## Inspect a training example

Preview the question, reference solution, and final numerical answer.

In [0]:
example = train[0]

print("QUESTION\n")
print(example["question"])
print("\nREFERENCE SOLUTION\n")
print(example["answer"])
print("\nFINAL ANSWER\n")
print(example["answer"].rsplit("####", 1)[-1].strip())

## Check duplicate examples and split overlap

Identify repeated questions and exact records within and across dataset splits.

In [0]:
def normalize_text(value):
    return " ".join((value or "").split()).casefold()


question_keys = {
    name: [normalize_text(row["question"]) for row in split]
    for name, split in dataset.items()
}

record_keys = {
    name: [
        (normalize_text(row["question"]), normalize_text(row["answer"]))
        for row in split
    ]
    for name, split in dataset.items()
}

print("Duplicates within splits:")

for name in dataset.keys():
    questions = question_keys[name]
    records = record_keys[name]
    print(f"{name}:")
    print(f"  Total examples: {len(questions):,}")
    print(f"  Unique questions: {len(set(questions)):,}")
    print(f"  Duplicate questions: {len(questions) - len(set(questions)):,}")
    print(f"  Duplicate records: {len(records) - len(set(records)):,}")

print("\nOverlap between splits:")

for name1, name2 in combinations(dataset.keys(), 2):
    overlap = set(question_keys[name1]) & set(question_keys[name2])
    record_overlap = set(record_keys[name1]) & set(record_keys[name2])
    print(f"{name1} vs {name2}:")
    print(f"  Shared unique questions: {len(overlap):,}")
    print(f"  {name1} examples involved: {sum(key in overlap for key in question_keys[name1]):,}")
    print(f"  {name2} examples involved: {sum(key in overlap for key in question_keys[name2]):,}")
    print(f"  Shared unique records: {len(record_overlap):,}")

all_questions = [key for keys in question_keys.values() for key in keys]
print(f"\nUnique questions across all splits: {len(set(all_questions)):,} / {len(all_questions):,}")

## Validate reference answers

Check final-answer markers and numerical targets.

In [0]:
NUMBER_PATTERN = re.compile(r"(?<![A-Za-z0-9])[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?")


def extract_final_text(solution):
    if "####" not in solution:
        return None
    return solution.rsplit("####", 1)[-1].strip()


def parse_final_number(solution):
    text = extract_final_text(solution)
    if text is None:
        return None
    try:
        return Decimal(text.replace(",", "").replace("$", ""))
    except InvalidOperation:
        return None


split_dataframes = {}

for split_name, split in dataset.items():
    records = []

    for row in split:
        question = row["question"]
        solution = row["answer"]
        final = parse_final_number(solution)

        records.append({
            "question": question,
            "solution": solution,
            "final_text": extract_final_text(solution),
            "final_number": float(final) if final is not None else np.nan,
            "question_words": len(question.split()),
            "solution_words": len(solution.split()),
            "question_numbers": len(NUMBER_PATTERN.findall(question)),
            "has_final_marker": "####" in solution,
        })

    split_dataframes[split_name] = pd.DataFrame(records)

for split_name, df in split_dataframes.items():
    print(f"\n{split_name.upper()}")
    print(f"Examples: {len(df):,}")
    print(f"Final-answer marker rate: {df['has_final_marker'].mean():.2%}")
    print(f"Numeric final-answer parse rate: {df['final_number'].notna().mean():.2%}")

    invalid = df[df["final_number"].isna()]
    if not invalid.empty:
        display(invalid[["question", "final_text"]].head(5))

## Measure problem and solution complexity

Summarize text lengths and numerical mentions.

In [0]:
columns = ["question_words", "solution_words", "question_numbers"]
statistics = ["mean", "std", "min", "50%", "95%", "max"]

for split_name, df in split_dataframes.items():
    print(f"\n{split_name.upper()}")

    summary = df[columns].describe(percentiles=[0.5, 0.95]).loc[statistics]
    summary = summary.rename(index={"50%": "median"})
    display(summary.round(2))

## Visualize the dataset

Compare split sizes, text lengths, annotated calculation steps, and final answers.

In [0]:

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(dataset.keys()), [len(dataset[name]) for name in dataset.keys()])
ax.set_title("Dataset Split Sizes")
ax.set_ylabel("Examples")
plt.tight_layout()
plt.show()

plots = [
    ("question_words", "Question Length", "Words"),
    ("solution_words", "Solution Length", "Words"),
    ("question_numbers", "Numerical Mentions in Questions", "Mentions"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (column, title, xlabel) in zip(axes.flat, plots):
    for split_name, df in split_dataframes.items():
        ax.hist(df[column], bins=30, alpha=0.5, label=split_name)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Examples")
    ax.legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))

for split_name, df in split_dataframes.items():
    values = np.log10(1 + df["final_number"].dropna().abs())
    ax.hist(values, bins=35, alpha=0.5, label=split_name)

ax.set_title("Final Answer Magnitudes")
ax.set_xlabel("log10(1 + |answer|)")
ax.set_ylabel("Examples")
ax.legend()
plt.tight_layout()
plt.show()


## Analyze arithmetic operations

Count operation symbols in annotated calculations for each split.

In [0]:
OPERATORS = {
    "Addition": "+",
    "Subtraction": "-",
    "Multiplication": "*",
    "Division": "/",
}

operation_counts = {}

for split_name, split in dataset.items():
    counts = Counter()

    for solution in split["answer"]:
        calculations = CALCULATION_PATTERN.findall(solution)
        for calculation in calculations:
            expression = calculation.split("=", 1)[0]
            for name, symbol in OPERATORS.items():
                counts[name] += expression.count(symbol)

    operation_counts[split_name] = counts

operation_frame = pd.DataFrame(operation_counts).fillna(0).astype(int)
display(operation_frame)

operation_frame.plot.bar(figsize=(9, 4), rot=0)
plt.title("Arithmetic Operators in Reference Calculations")
plt.ylabel("Occurrences")
plt.tight_layout()
plt.show()